# 05. Annual model: external test and paired comparison

Scores the frozen annual model from Training 04 once on JECAM, each supplemental campaign and the combined pool, in the main testing layout. The paired comparison then scores it against each window model of the current Testing 01 run on the parcels that model scored, with the parcel bootstrap of the external report (10,000 resamples, seed 42).

These external parcels were already scored for the window models, so the comparison is retrospective evidence rather than a fresh held-out test. Running this notebook publishes a new `current_test.json` under `pipeline_outputs/annual/`.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from IPython.display import Markdown, display
from crop_pipeline.analysis_workflow import resolve_stage_pointer
from crop_pipeline.training import annual_workflow, paired_comparison

# The window models' current external test (Testing 01) on this machine is the baseline.
baseline_run = resolve_stage_pointer(root / "pipeline_outputs/testing/current_scores.json").parent
print("Window-model testing run:", baseline_run.name)

In [ ]:
test = annual_workflow.test(root)
run = test.parent
for scope in json.loads((run / "reports/test_scopes.json").read_text()):
    folder = run / "reports" / ("" if scope == "jecam" else scope)
    scores = json.loads((folder / "test_scores.json").read_text())["annual"]
    print(f"{scope:40} parcels {sum(c['parcels'] for c in scores['coverage'].values()):4}  "
          f"macro-F1 {scores.get('macro_f1', 'unavailable')}")

In [ ]:
for revised in (False, True):
    output = root / "pipeline_outputs/reports" / f"annual_vs_windows_{run.name}{'_revised' if revised else ''}"
    if not output.exists():
        paired_comparison.report(run, "annual", baseline_run, output, revised=revised)
display(Markdown((root / "pipeline_outputs/reports" / f"annual_vs_windows_{run.name}" / "report.md").read_text()))